# Singular Value Decomposition via Eigendecomposition

This notebook demonstrates how to compute the Singular Value Decomposition (SVD) of a matrix using the eigendecomposition of $X^T X$ and $X X^T$, along with resolving eigenvector sign ambiguity and comparing performance against `numpy.linalg.svd`.

## Implementation & Function Definitions

In [ ]:
import numpy as np
from scipy import linalg


def eig_svd(X: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Computes the Singular Value Decomposition (SVD) of matrix X using 
    eigendecomposition of X^T @ X and X @ X^T.

    References:
        - Sign ambiguity in SVD: https://stackoverflow.com/a/52488038
        - SVD vs Eigendecomposition stability: https://math.stackexchange.com/q/4385842
        - Eigenvalue stability: https://math.stackexchange.com/q/3597333

    Args:
        X: Input array of shape (m, n).

    Returns:
        A tuple of (U, s, Vt) matching numpy.linalg.svd output format.
    """
    m, n = X.shape

    # Compute covariance matrices
    XtX = X.T @ X
    XXt = X @ X.T

    # Eigenvalues & Right singular vectors (V)
    s2_v, V = np.linalg.eig(XtX)
    order_v = np.argsort(s2_v)[::-1]
    V = V[:, order_v]

    # Eigenvalues & Left singular vectors (U)
    s2_u, U = np.linalg.eig(XXt)
    order_u = np.argsort(s2_u)[::-1]
    U = U[:, order_u]

    # Extract singular values based on matrix dimensions
    if m > n:
        s = np.sqrt(np.maximum(0, np.real(s2_v[order_v])))
    else:
        s = np.sqrt(np.maximum(0, np.real(s2_u[order_u])))

    return U.real, s, V.T.real


def align_signs(eig_vec: np.ndarray, ref_vec: np.ndarray) -> np.ndarray:
    """Aligns the sign orientation of eigenvectors with a reference matrix.

    Because U and V are calculated independently via eigendecomposition, their sign
    orientations are arbitrary. This function corrects signs relative to standard SVD.
    """
    return np.abs(eig_vec) * np.sign(ref_vec)

## Example & Validation

In [ ]:
# Generate random test matrix
np.random.seed(42)
X = np.random.randn(20, 4)
m, n = X.shape

# 1. Custom Eigendecomposition-based SVD
u_custom, s_custom, vt_custom = eig_svd(X)

# 2. Standard NumPy SVD (Reference)
u_ref, s_ref, vt_ref = np.linalg.svd(X)

# 3. Align sign orientation of custom components with reference SVD
u_aligned = align_signs(u_custom, u_ref)
vt_aligned = align_signs(vt_custom, vt_ref)

# Reconstruct original matrix
S_custom_matrix = linalg.diagsvd(s_custom, m, n)
S_ref_matrix = linalg.diagsvd(s_ref, m, n)

X_recon_custom = u_aligned @ S_custom_matrix @ vt_aligned
X_recon_ref = u_ref @ S_ref_matrix @ vt_ref

# Compute reconstruction error (Frobenius norm)
error_custom = np.linalg.norm(X_recon_custom - X)
error_ref = np.linalg.norm(X_recon_ref - X)

print(f"Custom Eig-SVD Reconstruction Error: {error_custom:.16e}")
print(f"Standard NumPy SVD Reconstruction Error: {error_ref:.16e}")